# Nebium on Kaggle

End-to-end: download Lichess `.zst` from YAML URLs, process once, cache the UCI corpus on a Hugging Face **dataset**, then train.

1. Enable **Internet**.
2. Add secrets: `HF_TOKEN` (required for dataset/model push), optional `WANDB_API_KEY`.
3. Replace `USER` in the train cell with your Hub username.

In [ ]:
import os
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
    try:
        os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    except Exception:
        os.environ["WANDB_MODE"] = "disabled"
except Exception:
    print("No Kaggle secrets client; set HF_TOKEN in the environment if you will push.")

REPO = Path("/kaggle/working/nebium")
if not (REPO / "scripts" / "train.py").exists():
    # Private GitHub: upload this repo as a Kaggle Dataset and copy it here instead.
    !git clone https://github.com/nabin2004/nebium.git {REPO}
%cd {REPO}
!pip install -e . --no-deps -q
!pip install chess zstandard hydra-core gguf wandb -q

In [ ]:
# First run: download URL(s) from configs/data/kaggle.yaml, process, push dataset, train.
# Later runs: pull USER/nebium-lichess-uci and skip PGN.
!python scripts/train.py --config-name kaggle \
  data.hf_dataset.repo_id=USER/nebium-lichess-uci \
  hub.repo_id=USER/nebium